# Logistic regression

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
Despite its name, logistic regression is a **classification** model. It first builds a score as linear regression does, then turns it into a probability between 0 and 1 with the **sigmoid** function:

$$z=b+w_1x_1+\dots+w_dx_d,\qquad p=\sigma(z)=\frac{1}{1+e^{-z}}$$

If $p\ge0.5$ (that is, $z\ge0$), predict class 1. So the decision boundary is $z=0$: a **line** (or plane).

[[lab:sigmoid]]

| $z$ | $\sigma(z)$ |
|---|---|
| −4 | 0.018 |
| −1 | 0.269 |
| 0 | 0.5 |
| 1 | 0.731 |
| 4 | 0.982 |

## Interpreting coefficients: odds ratios
$z$ is the log of the **odds**: $z=\ln\frac{p}{1-p}$. If $w_j=0.7$, one more unit of $x_j$ multiplies the odds by $e^{0.7}\approx2$.

## The cost: cross-entropy
MSE does not suit probabilities. We use **log loss**, which heavily punishes confident wrong predictions:

$$J=-\frac1n\sum_i\big[y_i\ln p_i+(1-y_i)\ln(1-p_i)\big]$$

> [Left: log loss; when the true class is 1 and the model says p≈0, the error goes to infinity. Right: with polynomial features the logistic boundary becomes curved.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/logit_loss.png)

*Left: log loss; when the true class is 1 and the model says p≈0, the error goes to infinity. Right: with polynomial features the logistic boundary becomes curved.*

Its gradient is simple: $\nabla_w J=\frac1n X^\top(p-y)$, minimized by gradient descent.

## A worked example
$w=2$, $b=-3$. For $x=2$: $z=1$ and $p=0.731$ → class 1. If the true label is 1, the loss is $-\ln0.731=0.313$; if it is 0, the loss is $-\ln0.269=1.313$.



In [ ]:
import numpy as np

# Logistic regression from scratch with gradient descent
rng = np.random.default_rng(0)
hours = rng.uniform(0, 10, 100)                       # study hours
passed = (hours + rng.normal(0, 1.5, 100) > 5).astype(int)

x = (hours - hours.mean()) / hours.std()              # scaling helps gradient descent
w, b, lr = 0.0, 0.0, 0.5
for step in range(500):
    p = 1 / (1 + np.exp(-(w * x + b)))
    w -= lr * np.mean((p - passed) * x)
    b -= lr * np.mean(p - passed)
loss = -np.mean(passed * np.log(p) + (1 - passed) * np.log(1 - p))
print(f"w = {w:.3f}, b = {b:.3f}, log loss = {loss:.3f}")
h = 6
print("P(pass | 6 hours) =", round(1 / (1 + np.exp(-(w * (h - hours.mean()) / hours.std() + b))), 3))
print("Odds multiply by", round(np.exp(w / hours.std()), 2), "for each extra hour")



## Decision boundary lab
With degree 1 the boundary is straight. On the "two moons" data, set the degree to 3. `C` is the inverse of the regularization strength: small C means strong regularization.

[[lab:cls_logistic]]

## Multi-class: softmax
For $K$ classes, we build one score $z_k$ per class and the probabilities are:

$$p_k=\frac{e^{z_k}}{\sum_j e^{z_j}}$$

| Hyperparameter | Meaning |
|---|---|
| `C` | Inverse regularization (default 1); smaller = simpler |
| `penalty` | `l2` (default), `l1` (feature selection), `elasticnet` |
| `class_weight` | `balanced` for imbalanced data |
| `max_iter` | Raise it if you get a convergence warning |



In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

iris = load_iris()
model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0)).fit(iris.data, iris.target)
print("Accuracy (5-fold):", cross_val_score(model, iris.data, iris.target, cv=5).mean().round(3))
probs = model.predict_proba(iris.data[[0, 75, 140]])
for row, name in zip(probs.round(3), ["sample 0", "sample 75", "sample 140"]):
    print(name, dict(zip(iris.target_names, row)), "sum =", row.sum().round(3))



| Pros | Cons |
|---|---|
| Well-calibrated probabilities | Only a linear boundary (unless you build features) |
| Fast and interpretable | Needs scaling |
| A great baseline | Unstable with correlated features |

### Exercise

Compute the probability and the log loss of the lesson's example (w=2, b=-3, x=2) with numpy.

In [ ]:
# Your answer here

### Exercise

On breast cancer, fit LogisticRegression with penalty='l1', solver='liblinear', C=0.1. How many coefficients are zero?

In [ ]:
# Your answer here